<a href="https://colab.research.google.com/github/shalini2004singh/ecommerce-delivery-analysis/blob/main/notebooks/03_findings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
!find /content/drive/MyDrive -iname "*olist*"

In [14]:
import pandas as pd

orders = pd.read_csv("/content/olist_orders_dataset.csv")
date_cols = ["order_purchase_timestamp", "order_approved_at",
             "order_delivered_carrier_date", "order_delivered_customer_date",
             "order_estimated_delivery_date"]
orders[date_cols] = orders[date_cols].apply(pd.to_datetime, errors="coerce")

print(orders.shape)
print(orders.isna().sum())
print(orders["order_id"].duplicated().sum())
print(orders["order_status"].value_counts())

(99441, 8)
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64
0
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


In [15]:
delivered = orders[orders["order_status"] == "delivered"]

print("Delivered but missing delivery date:", delivered["order_delivered_customer_date"].isna().sum())
print("Delivered before purchase:", (orders["order_delivered_customer_date"] < orders["order_purchase_timestamp"]).sum())
print("Carrier date before purchase:", (orders["order_delivered_carrier_date"] < orders["order_purchase_timestamp"]).sum())
print("Delivered to customer before carrier:", (orders["order_delivered_customer_date"] < orders["order_delivered_carrier_date"]).sum())
print("Purchase date range:", orders["order_purchase_timestamp"].min(), "to", orders["order_purchase_timestamp"].max())

days = (orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]).dt.days
print(days.describe())

Delivered but missing delivery date: 8
Delivered before purchase: 0
Carrier date before purchase: 166
Delivered to customer before carrier: 23
Purchase date range: 2016-09-04 21:15:19 to 2018-10-17 17:30:18
count    96476.000000
mean        12.094086
std          9.551746
min          0.000000
25%          6.000000
50%         10.000000
75%         15.000000
max        209.000000
dtype: float64


In [16]:
import os
os.makedirs("/content/interim", exist_ok=True)

orders["flag_missing_delivery_date"] = (orders["order_status"] == "delivered") & orders["order_delivered_customer_date"].isna()
orders["flag_bad_carrier_ts"] = orders["order_delivered_carrier_date"] < orders["order_purchase_timestamp"]
orders["flag_delivered_before_carrier"] = orders["order_delivered_customer_date"] < orders["order_delivered_carrier_date"]

orders.to_csv("/content/interim/orders_flagged.csv", index=False)
print(orders[["flag_missing_delivery_date", "flag_bad_carrier_ts", "flag_delivered_before_carrier"]].sum())

flag_missing_delivery_date         8
flag_bad_carrier_ts              166
flag_delivered_before_carrier     23
dtype: int64


In [17]:
d = orders[orders["order_status"] == "delivered"].copy()
d = d[d["order_delivered_customer_date"].notna()]

d["delivery_days"] = (d["order_delivered_customer_date"] - d["order_purchase_timestamp"]).dt.days
d["delay_days"] = (d["order_delivered_customer_date"] - d["order_estimated_delivery_date"]).dt.total_seconds() / 86400
d["is_late"] = d["order_delivered_customer_date"] > d["order_estimated_delivery_date"]

d["processing_days"] = (d["order_delivered_carrier_date"] - d["order_purchase_timestamp"]).dt.total_seconds() / 86400
d.loc[d["flag_bad_carrier_ts"], "processing_days"] = None

print("Delivered orders used:", len(d))
print("On-time rate %:", round((1 - d["is_late"].mean()) * 100, 2))
print("Cancellation rate %:", round((orders["order_status"] == "canceled").mean() * 100, 2))
print("Avg delivery days:", round(d["delivery_days"].mean(), 2))
print("Median delivery days:", d["delivery_days"].median())
print("Avg delay days (all orders, negative = early):", round(d["delay_days"].mean(), 2))
print("Avg delay days (late orders only):", round(d.loc[d["is_late"], "delay_days"].mean(), 2))
print("Avg processing days:", round(d["processing_days"].mean(), 2))

Delivered orders used: 96470
On-time rate %: 91.89
Cancellation rate %: 0.63
Avg delivery days: 12.09
Median delivery days: 10.0
Avg delay days (all orders, negative = early): -11.18
Avg delay days (late orders only): 9.55
Avg processing days: 3.23


In [18]:
customers = pd.read_csv("/content/olist_customers_dataset.csv")
d = d.merge(customers[["customer_id", "customer_state"]], on="customer_id", how="left")

by_state = d.groupby("customer_state").agg(
    orders=("order_id", "count"),
    on_time_rate=("is_late", lambda x: round((1 - x.mean()) * 100, 2)),
    avg_delivery_days=("delivery_days", "mean"),
).round(2).sort_values("on_time_rate")

print(by_state)

                orders  on_time_rate  avg_delivery_days
customer_state                                         
AL                 397         76.07              24.04
MA                 717         80.33              21.12
PI                 476         84.03              18.99
CE                1279         84.68              20.82
SE                 335         84.78              21.03
BA                3256         85.96              18.87
RJ               12350         86.53              14.85
TO                 274         87.23              17.23
PA                 946         87.63              23.32
ES                1995         87.77              15.33
RR                  41         87.80              28.98
MS                 701         88.45              15.19
PB                 517         88.97              19.95
PE                1593         89.20              17.97
RN                 474         89.24              18.82
SC                3546         90.24            

In [19]:
d["purchase_weekday"] = d["order_purchase_timestamp"].dt.day_name()

by_day = d.groupby("purchase_weekday").agg(
    orders=("order_id", "count"),
    on_time_rate=("is_late", lambda x: round((1 - x.mean()) * 100, 2)),
    avg_delivery_days=("delivery_days", "mean"),
).round(2)

print(by_day.reindex(["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]))

                  orders  on_time_rate  avg_delivery_days
purchase_weekday                                         
Monday             15701         90.95              11.53
Tuesday            15502         91.51              11.61
Wednesday          15074         92.20              11.98
Thursday           14322         92.43              12.28
Friday             13684         91.55              13.12
Saturday           10555         92.40              12.89
Sunday             11632         92.51              11.47


In [20]:
reviews = pd.read_csv("/content/olist_order_reviews_dataset.csv")

# some orders have more than one review, so keep one per order (the latest)
reviews = reviews.sort_values("review_creation_date").drop_duplicates("order_id", keep="last")

r = d.merge(reviews[["order_id", "review_score"]], on="order_id", how="left")

print("Orders with a review:", r["review_score"].notna().sum(), "of", len(r))
print(r.groupby("is_late")["review_score"].agg(["count", "mean", "median"]).round(2))
print(r.groupby("is_late")["review_score"].apply(lambda x: (x <= 2).mean() * 100).round(2))

Orders with a review: 95824 of 96470
         count  mean  median
is_late                     
False    88163  4.29     5.0
True      7661  2.57     2.0
is_late
False     9.17
True     52.91
Name: review_score, dtype: float64


In [21]:
!wc -l /content/olist_orders_dataset.csv

99442 /content/olist_orders_dataset.csv


In [22]:
print(len(orders), len(d), len(reviews))

99441 96470 98673


In [23]:
items = pd.read_csv("/content/olist_order_items_dataset.csv")
products = pd.read_csv("/content/olist_products_dataset.csv")
trans = pd.read_csv("/content/product_category_name_translation.csv")
sellers = pd.read_csv("/content/olist_sellers_dataset.csv")

first_item = items.drop_duplicates("order_id")[["order_id", "product_id", "seller_id"]]

d2 = (d.merge(first_item, on="order_id", how="left")
       .merge(products[["product_id", "product_category_name"]], on="product_id", how="left")
       .merge(trans, on="product_category_name", how="left")
       .merge(sellers[["seller_id", "seller_state"]], on="seller_id", how="left")
       .merge(reviews[["order_id", "review_score"]], on="order_id", how="left"))

print(len(d2), "rows. This should equal", len(d))

96470 rows. This should equal 96470


In [25]:
by_cat = d2.groupby("product_category_name_english").agg(
    orders=("order_id", "count"),
    on_time_rate=("is_late", lambda x: round((1 - x.mean()) * 100, 2)),
    avg_delivery_days=("delivery_days", "mean"),
    avg_review=("review_score", "mean"),
).round(2)

print(by_cat[by_cat["orders"] >= 200].sort_values("on_time_rate").head(10))

                                 orders  on_time_rate  avg_delivery_days  \
product_category_name_english                                              
audio                               344         86.92              13.07   
books_technical                     255         89.02              10.22   
home_confort                        370         89.73              13.08   
food                                435         89.89               9.37   
electronics                        2507         90.15              12.47   
baby                               2763         90.66              12.28   
office_furniture                   1246         90.77              20.23   
construction_tools_construction     728         90.80              10.42   
construction_tools_lights           231         90.91               9.28   
health_beauty                      8608         91.01              11.64   

                                 avg_review  
product_category_name_english            

In [26]:
d2["same_state"] = d2["customer_state"] == d2["seller_state"]

print(d2.groupby("same_state").agg(
    orders=("order_id", "count"),
    on_time_rate=("is_late", lambda x: round((1 - x.mean()) * 100, 2)),
    avg_delivery_days=("delivery_days", "mean"),
).round(2))

by_seller_state = d2.groupby("seller_state").agg(
    orders=("order_id", "count"),
    on_time_rate=("is_late", lambda x: round((1 - x.mean()) * 100, 2)),
).sort_values("orders", ascending=False).head(8)
print(by_seller_state)

            orders  on_time_rate  avg_delivery_days
same_state                                         
False        61769         90.73              14.68
True         34701         93.94               7.48
              orders  on_time_rate
seller_state                      
SP             68415         91.23
MG              7647         94.36
PR              7429         93.47
RJ              4185         91.54
SC              3551         93.95
RS              1949         95.69
DF               803         94.02
BA               549         94.17


In [27]:
d2["transit_days"] = (d2["order_delivered_customer_date"] - d2["order_delivered_carrier_date"]).dt.total_seconds() / 86400
d2.loc[d2["flag_bad_carrier_ts"] | d2["flag_delivered_before_carrier"], "transit_days"] = None

print(d2.groupby("is_late")[["processing_days", "transit_days"]].mean().round(2))

         processing_days  transit_days
is_late                               
False               3.00          7.89
True                5.84         25.68


In [28]:
d2["month"] = d2["order_purchase_timestamp"].dt.to_period("M")
trend = d2.groupby("month").agg(
    orders=("order_id", "count"),
    on_time_rate=("is_late", lambda x: round((1 - x.mean()) * 100, 2)),
).loc["2017-01":"2018-08"]
print(trend)

         orders  on_time_rate
month                        
2017-01     750         96.93
2017-02    1653         96.79
2017-03    2546         94.42
2017-04    2303         92.14
2017-05    3545         96.39
2017-06    3135         96.14
2017-07    3872         96.57
2017-08    4193         96.68
2017-09    4150         94.80
2017-10    4478         94.71
2017-11    7288         85.69
2017-12    5513         91.62
2018-01    7069         93.44
2018-02    6555         84.01
2018-03    7003         78.64
2018-04    6798         94.69
2018-05    6749         91.76
2018-06    6096         98.64
2018-07    6156         95.52
2018-08    6351         89.61


In [29]:
st = d2.groupby("customer_state").agg(
    orders=("order_id", "count"),
    on_time_rate=("is_late", lambda x: (1 - x.mean()) * 100),
)
st["late_orders"] = (st["orders"] * (100 - st["on_time_rate"]) / 100).round()
st["low_sample"] = st["orders"] < 100
print(st.sort_values("late_orders", ascending=False).head(8).round(2))

                orders  on_time_rate  late_orders  low_sample
customer_state                                               
SP               40494         94.11       2387.0       False
RJ               12350         86.53       1664.0       False
MG               11354         94.39        637.0       False
BA                3256         85.96        457.0       False
RS                5344         92.85        382.0       False
SC                3546         90.24        346.0       False
PR                4923         95.00        246.0       False
ES                1995         87.77        244.0       False


In [30]:
os.makedirs("/content/processed", exist_ok=True)
out = d2[["order_id", "order_purchase_timestamp", "order_estimated_delivery_date",
          "order_delivered_customer_date", "customer_state", "seller_state",
          "product_category_name_english", "delivery_days", "delay_days",
          "is_late", "processing_days", "transit_days", "review_score", "same_state"]]
out.to_csv("/content/processed/orders_analytical.csv", index=False)

from google.colab import files
files.download("/content/processed/orders_analytical.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>